## Домашнее задание: машинный перевод с помощью многоязычных эмбеддингов (6 баллов)

> Неофициальный русский перевод [исходного задания](https://github.com/yandexdataschool/nlp_course/blob/2026/week01_embeddings/homework.ipynb). Исполняемый код и автоматические проверки сохранены.

**В этом домашнем задании** **<font color='red'>ТЫ</font>** построишь систему машинного перевода без параллельных корпусов, выравнивания, механизма внимания, глубоких рекуррентных нейросетей и других сложных компонентов.

Даже без параллельных корпусов такой переводчик может работать достаточно хорошо (надеемся!).

Для эксперимента возьмём два родственных славянских языка: украинский и русский.

### Почувствуй разницу!

(_синій кіт_ и _синій кит_)

![Синий кот и синий кит](https://github.com/yandexdataschool/nlp_course/raw/master/resources/blue_cat_blue_whale.png)

### Фрагмент списка Сводеша для нескольких славянских языков

Список Сводеша — инструмент лексикостатистики. Он назван в честь американского лингвиста Морриса Сводеша и содержит базовую лексику. Такие списки используются для определения родства языков и их классификации.

Видно, что некоторые слова разных славянских языков имеют сходную форму.

| Русский         | Белорусский              | Украинский              | Польский           | Чешский                       | Болгарский            |
|-----------------|--------------------------|-------------------------|--------------------|-------------------------------|-----------------------|
| женщина         | жанчына, кабета, баба    | жінка                   | kobieta            | žena                          | жена                  |
| мужчина         | мужчына                  | чоловік, мужчина        | mężczyzna          | muž                           | мъж                   |
| человек         | чалавек                  | людина, чоловік         | człowiek           | člověk                        | човек                 |
| ребёнок, дитя   | дзіця, дзіцёнак, немаўля | дитина, дитя            | dziecko            | dítě                          | дете                  |
| жена            | жонка                    | дружина, жінка          | żona               | žena, manželka, choť          | съпруга, жена         |
| муж             | муж, гаспадар            | чоловiк, муж            | mąż                | muž, manžel, choť             | съпруг, мъж           |
| мать, мама      | маці, матка              | мати, матір, неня, мама | matka              | matka, máma, 'стар.' mateř    | майка                 |
| отец, тятя      | бацька, тата             | батько, тато, татусь    | ojciec             | otec                          | баща, татко           |
| много           | шмат, багата             | багато                  | wiele              | mnoho, hodně                  | много                 |
| несколько       | некалькі, колькі         | декілька, кілька        | kilka              | několik, pár, trocha          | няколко               |
| другой, иной    | іншы                     | інший                   | inny               | druhý, jiný                   | друг                  |
| зверь, животное | жывёла, звер, істота     | тварина, звір           | zwierzę            | zvíře                         | животно               |
| рыба            | рыба                     | риба                    | ryba               | ryba                          | риба                  |
| птица           | птушка                   | птах, птиця             | ptak               | pták                          | птица                 |
| собака, пёс     | сабака                   | собака, пес             | pies               | pes                           | куче, пес             |
| вошь            | вош                      | воша                    | wesz               | veš                           | въшка                 |
| змея, гад       | змяя                     | змія, гад               | wąż                | had                           | змия                  |
| червь, червяк   | чарвяк                   | хробак, черв'як         | robak              | červ                          | червей                |
| дерево          | дрэва                    | дерево                  | drzewo             | strom, dřevo                  | дърво                 |
| лес             | лес                      | ліс                     | las                | les                           | гора, лес             |
| палка           | кій, палка               | палиця                  | patyk, pręt, pałka | hůl, klacek, prut, kůl, pálka | палка, пръчка, бастун |

Но ещё более устойчивыми между этими языками оказываются распределения контекстов. Это свойство мы и используем.

## Данные

In [ ]:
import gensim
import numpy as np
from gensim.models import KeyedVectors

Скачай готовые векторные представления слов:
- [Украинские эмбеддинги — cc.uk.300.vec.zip](https://yadi.sk/d/9CAeNsJiInoyUA)
- [Русские эмбеддинги — cc.ru.300.vec.zip](https://yadi.sk/d/3yG0-M4M8fypeQ)

Загрузи эмбеддинги украинского и русского языков.

In [ ]:
uk_emb = KeyedVectors.load_word2vec_format("cc.uk.300.vec")

In [ ]:
ru_emb = KeyedVectors.load_word2vec_format("cc.ru.300.vec")

assert uk_emb.vector_size == ru_emb.vector_size == 300

In [ ]:
ru_emb.most_similar([ru_emb["август"]], topn=10)

In [ ]:
uk_emb.most_similar([uk_emb["серпень"]])

In [ ]:
ru_emb.most_similar([uk_emb["серпень"]])

Загрузи небольшие двуязычные словари соответствующих пар слов для обучения и тестирования.

> **Покрытие словаря.** В файлах `.vec` хранятся векторы только для конечного множества слов. В отличие от полной модели fastText, объект `KeyedVectors`, загруженный из `.vec`, **не может генерировать векторы для отсутствующих слов**. Поэтому пропуски неизбежны. При загрузке двуязычных словарей пропускай пару, если хотя бы одно слово отсутствует в соответствующем словаре эмбеддингов. Приведённый ниже загрузчик выводит количество сохранённых и пропущенных пар.

In [ ]:
def load_word_pairs(filename):
    uk_ru_pairs = []
    uk_vectors = []
    ru_vectors = []
    num_skipped = 0
    with open(filename, "r", encoding="utf-8") as inpf:
        for line in inpf:
            uk, ru = line.rstrip().split("\t")
            if uk not in uk_emb or ru not in ru_emb:
                num_skipped += 1
                continue
            uk_ru_pairs.append((uk, ru))
            uk_vectors.append(uk_emb[uk])
            ru_vectors.append(ru_emb[ru])
    print(f"{filename}: сохранено {len(uk_ru_pairs)} пар, пропущено {num_skipped} OOV-пар")
    return uk_ru_pairs, np.array(uk_vectors), np.array(ru_vectors)

In [ ]:
uk_ru_train, X_train, Y_train = load_word_pairs("ukr_rus.train.txt")

In [ ]:
uk_ru_test, X_test, Y_test = load_word_pairs("ukr_rus.test.txt")

## Преобразование между пространствами эмбеддингов

Пусть $x_i \in \mathrm{R}^d$ — вектор-строка слова $i$ в исходном языке, а $y_i \in \mathrm{R}^d$ — вектор-строка его перевода. Наша цель — научиться линейному преобразованию $W$, которое минимизирует квадрат евклидова расстояния между $x_iW$ и $y_i$ для набора пар слов.

Задачу можно записать как метод наименьших квадратов:

$$W^*= \arg\min_W \sum_{i=1}^n||x_iW - y_i||_2^2$$

или:

$$W^*= \arg\min_W ||XW - Y||_F^2$$

Здесь строки матриц $X$ и $Y$ содержат эмбеддинги исходных слов и их переводов, а $||*||_F$ — **норма Фробениуса**.

В греческой мифологии Прокруст — разбойник из Аттики, который укладывал путников на железное ложе и подгонял их рост под его длину, растягивая или отрубая ноги. Мы похожим образом «подгоняем» пространство исходных эмбеддингов под целевое пространство. Наше прокрустово ложе — пространство эмбеддингов русского языка.

![Отображение между пространствами эмбеддингов](https://github.com/yandexdataschool/nlp_course/raw/master/resources/embedding_mapping.png)

*На рисунке использована запись с векторами-столбцами $Wx$. В этом ноутбуке эмбеддинги хранятся строками, поэтому соответствующая операция для пакета векторов — $XW$.*

![Ортогональная задача Прокруста](https://github.com/yandexdataschool/nlp_course/raw/master/resources/procrustes.png)

Но постой: выражение $W^*= \arg\min_W \sum_{i=1}^n||x_iW - y_i||_2^2$ похоже на обычную многомерную линейную регрессию (без свободного члена). Давай реализуем его.

In [ ]:
from sklearn.linear_model import LinearRegression

# ТВОЙ КОД ЗДЕСЬ
# Пример: mapping = LinearRegression(fit_intercept=False).fit(X_train, Y_train)

Посмотрим на ближайших соседей вектора украинского слова _«серпень»_ (_«август»_ по-русски) после линейного преобразования.

In [ ]:
august = mapping.predict(uk_emb["серпень"].reshape(1, -1))
ru_emb.most_similar(august)

Видим, что ближайшие соседи полученного эмбеддинга — названия разных месяцев, однако правильный перевод стоит только на девятом месте.

Для оценки качества используем **precision@1, precision@5 и precision@10**: для каждого преобразованного украинского эмбеддинга проверяем, находится ли правильный русский перевод среди $N$ ближайших соседей в русском векторном пространстве.

In [ ]:
def precision(pairs, mapped_vectors, topn=1):
    """
    :args:
        pairs — список правильных пар слов [(uk_word_0, ru_word_0), ...]
        mapped_vectors — эмбеддинги после преобразования из исходного в целевое пространство
        topn — сколько ближайших соседей учитывать в целевом пространстве
    :returns:
        precision_val — доля слов, для которых верный перевод вошёл в top-K ближайших соседей.
    """
    assert len(pairs) == len(mapped_vectors)
    num_matches = 0
    for i, (_, ru) in enumerate(pairs):
        # ТВОЙ КОД ЗДЕСЬ
        pass
    precision_val = num_matches / len(pairs)
    return precision_val


In [ ]:
assert precision([("серпень", "август")], august, topn=5) == 0.0
assert precision([("серпень", "август")], august, topn=9) == 1.0
assert precision([("серпень", "август")], august, topn=10) == 1.0

In [ ]:
assert precision(uk_ru_test, X_test) == 0.0
assert precision(uk_ru_test, Y_test) == 1.0

In [ ]:
precision_top1 = precision(uk_ru_test, mapping.predict(X_test), 1)
precision_top5 = precision(uk_ru_test, mapping.predict(X_test), 5)

assert precision_top1 >= 0.635
assert precision_top5 >= 0.813

## Улучшаем результат: ортогональная задача Прокруста

Можно показать (см. [исходную статью](https://arxiv.org/pdf/1710.04087)), что согласованное линейное отображение между семантическими пространствами должно быть ортогональным.

Ограничим преобразование $W$ условием ортогональности. Тогда задача принимает вид:

$$W^*= \arg\min_W ||XW - Y||_F^2 \text{, где: } W^TW = I$$

$$I \text{ — единичная матрица}$$

Вместо решения ещё одной задачи регрессии найдём оптимальное ортогональное преобразование через **сингулярное разложение (SVD)**. Оказывается, оптимальное $W^*$ выражается через компоненты SVD:

$$X^TY=U\Sigma V^T\text{, сингулярное разложение}$$

$$W^*=UV^T$$

In [ ]:
def learn_transform(X_train, Y_train):
    """ 
    :returns: W* — матрица float[emb_dim × emb_dim], определённая формулами выше
    """
    # ТВОЙ КОД ЗДЕСЬ
    pass

In [ ]:
W = learn_transform(X_train, Y_train)

In [ ]:
ru_emb.most_similar([np.matmul(uk_emb["серпень"], W)])

In [ ]:
assert precision(uk_ru_test, np.matmul(X_test, W)) >= 0.653
assert precision(uk_ru_test, np.matmul(X_test, W), 5) >= 0.824

## Переводчик с украинского на русский

Теперь можно построить простой пословный переводчик: для каждого слова исходного языка ищем ближайший вектор в пространстве эмбеддингов целевого языка после преобразования.

### Правила обработки слов вне словаря (OOV)

Функция `translate` получает предложение **в нижнем регистре** и разбивает его на токены по пробелам с помощью `sentence.split()`. Для каждого токена:

- Если токен **не содержит букв** (`any(char.isalpha() for char in token)` возвращает `False`), скопируй его без изменений, даже если его нет в словаре эмбеддингов.
- Иначе, если токен есть в `uk_emb`, переведи его через выученное преобразование.
- В остальных случаях выведи **буквально** строку `<UNK>`.

`<UNK>` — маркер в выходном тексте; ему не требуется собственный эмбеддинг. Словарь конечен, поэтому неизвестные украинские слова — нормальная ситуация: они не должны приводить к исключению `KeyError` или оставаться без перевода.

Для этого задания предложенная сказка **не токенизируется повторно**. Если знак препинания примыкает к слову, вся последовательность остаётся одним токеном и обрабатывается по правилам OOV. Поэтому несколько `<UNK>` в итоговом переводе — ожидаемое поведение. Эти правила обязательны для автоматической проверки.

In [ ]:
with open("fairy_tale.txt", "r", encoding="utf-8") as inpf:
    uk_sentences = [line.rstrip().lower() for line in inpf]

In [ ]:
def translate(sentence):
    """
    :args:
        sentence — украинское предложение в нижнем регистре (str)
    :returns:
        translation — предложение на русском (str)

    * сохраняй пунктуацию и числа без изменений
    * переводи известные слова через обученное преобразование
    * заменяй неизвестные слова на <UNK>
    """
    # ТВОЙ КОД ЗДЕСЬ
    pass

In [ ]:
oov_word = "цьогословаточнонемаєвсловнику"
assert oov_word not in uk_emb
assert translate(oov_word) == "<UNK>"
assert translate(f"кіт {oov_word} !") == "кот <UNK> !"
assert translate(".") == "."
assert translate("1 , 3") == "1 , 3"
assert translate("кіт зловив мишу") == "кот поймал мышку"

In [ ]:
for sentence in uk_sentences:
    print("src: {}\ndst: {}\n".format(sentence, translate(sentence)))

Неплохо, правда? Перевод можно улучшить с помощью языковой модели и выбора не одного, а нескольких ближайших соседей в общем векторном пространстве. Но это уже тема следующего занятия.

## Что почитать дальше?

### Статьи
- [Exploiting Similarities among Languages for Machine Translation](https://arxiv.org/pdf/1309.4168) — введение в многоязычные эмбеддинги от Томаша Миколова (автора Word2Vec).
- [Offline bilingual word vectors, orthogonal transformations and the inverted softmax](https://arxiv.org/pdf/1702.03859) — ортогональное преобразование для машинного перевода без параллельного корпуса.
- [Word Translation Without Parallel Data](https://arxiv.org/pdf/1710.04087).
- [Loss in Translation: Learning Bilingual Word Mapping with a Retrieval Criterion](https://arxiv.org/pdf/1804.07745).
- [Unsupervised Alignment of Embeddings with Wasserstein Procrustes](https://arxiv.org/abs/1805.11222).

### Репозитории с готовыми многоязычными эмбеддингами
- https://github.com/facebookresearch/MUSE
- https://github.com/Babylonpartners/fastText_multilingual